[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kitoware/cv-project/blob/main/emotion_recognition.ipynb)

# Video-Based Emotion Recognition using CNN+LSTM on RAVDESS

This notebook implements a video-based emotion recognition system that classifies facial expressions into 8 emotional categories using temporal modeling. The pipeline:

1. **Face Detection & Alignment** — MTCNN detects and crops faces from video frames
2. **Spatial Feature Extraction** — Pretrained ResNet-18 extracts 512-d features per frame
3. **Temporal Modeling** — LSTM captures expression dynamics across 15 sampled frames
4. **Emotion Classification** — Fully connected layer outputs 8 emotion class probabilities

**Dataset:** [RAVDESS](https://zenodo.org/records/1188976) — 24 actors, 8 emotions, ~1440 speech videos

---

### Table of Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Dataset Download & Organization](#2-dataset-download--organization)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Face Detection & Preprocessing](#4-face-detection--preprocessing)
5. [Dataset & DataLoader](#5-dataset--dataloader)
6. [Model Architecture](#6-model-architecture)
7. [Training](#7-training)
8. [Evaluation](#8-evaluation)
9. [Qualitative Analysis](#9-qualitative-analysis)
10. [Discussion & Conclusion](#10-discussion--conclusion)

## 1. Setup & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q facenet-pytorch --no-deps

In [ ]:
import os
import glob
import random
import zipfile
import requests
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from facenet_pytorch import MTCNN

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE
from tqdm.auto import tqdm

CONFIG = {
    'num_frames': 15,
    'face_size': 224,
    'batch_size': 8,
    'num_epochs': 30,
    'learning_rate': 1e-4,
    'cnn_learning_rate': 1e-5,
    'weight_decay': 1e-3,
    'lstm_hidden': 256,
    'lstm_layers': 2,
    'num_classes': 8,
    'dropout': 0.5,
    'grad_clip': 1.0,
    'early_stop_patience': 7,
    'seed': 42,
}

BASE_DIR = Path('/content/drive/MyDrive/cv-project')
DATA_DIR = BASE_DIR / 'data' / 'ravdess'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'

for d in [DATA_DIR, PROCESSED_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
torch.cuda.manual_seed_all(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

EMOTION_MAP = {
    1: 'neutral',
    2: 'calm',
    3: 'happy',
    4: 'sad',
    5: 'angry',
    6: 'fearful',
    7: 'disgust',
    8: 'surprised',
}
EMOTION_NAMES = [EMOTION_MAP[i] for i in range(1, 9)]

## 2. Dataset Download & Organization

The [RAVDESS dataset](https://zenodo.org/records/1188976) contains 7,356 files from 24 professional actors (12 male, 12 female). Each filename encodes 7 attributes:

| Position | Attribute | Values |
|----------|-----------|--------|
| 1 | Modality | 01=full-AV, 02=video-only, 03=audio-only |
| 2 | Vocal channel | 01=speech, 02=song |
| 3 | Emotion | 01-08 (see mapping above) |
| 4 | Intensity | 01=normal, 02=strong |
| 5 | Statement | 01="Kids are talking...", 02="Dogs are sitting..." |
| 6 | Repetition | 01=1st, 02=2nd |
| 7 | Actor | 01-24 (odd=male, even=female) |

We download only the **speech video** files (24 actor zips).

In [ ]:
def download_ravdess(data_dir, num_actors=24):
    """Download RAVDESS speech video files from Zenodo."""
    base_url = 'https://zenodo.org/records/1188976/files'
    
    for actor_id in tqdm(range(1, num_actors + 1), desc='Downloading actors'):
        zip_name = f'Video_Speech_Actor_{actor_id:02d}.zip'
        zip_path = data_dir / zip_name
        actor_dir = data_dir / f'Actor_{actor_id:02d}'
        
        if actor_dir.exists() and any(actor_dir.glob('*.mp4')):
            continue
        
        if not zip_path.exists():
            url = f'{base_url}/{zip_name}?download=1'
            response = requests.get(url, stream=True)
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            
            with open(zip_path, 'wb') as f:
                with tqdm(total=total, unit='B', unit_scale=True, 
                          desc=zip_name, leave=False) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(data_dir)
        
        zip_path.unlink()

download_ravdess(DATA_DIR)

In [ ]:
def parse_ravdess_filename(filepath):
    """Parse RAVDESS filename into metadata dict."""
    name = Path(filepath).stem
    parts = name.split('-')
    return {
        'filepath': str(filepath),
        'modality': int(parts[0]),
        'vocal_channel': int(parts[1]),
        'emotion': int(parts[2]),
        'intensity': int(parts[3]),
        'statement': int(parts[4]),
        'repetition': int(parts[5]),
        'actor': int(parts[6]),
    }

all_files = sorted(glob.glob(str(DATA_DIR / 'Actor_*' / '*.mp4')))
metadata = pd.DataFrame([parse_ravdess_filename(f) for f in all_files])

# Filter: speech only (vocal_channel=1), video modalities only (1=full-AV, 2=video-only)
metadata = metadata[
    (metadata['vocal_channel'] == 1) & 
    (metadata['modality'].isin([1, 2]))
].reset_index(drop=True)

metadata['emotion_label'] = metadata['emotion'].map(EMOTION_MAP)
metadata['gender'] = metadata['actor'].apply(lambda x: 'male' if x % 2 == 1 else 'female')
# 0-indexed label for PyTorch
metadata['label'] = metadata['emotion'] - 1

print(f'Total speech video samples: {len(metadata)}')
print(f'\nEmotion distribution:')
print(metadata['emotion_label'].value_counts().sort_index())
metadata.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=metadata, x='emotion_label', hue='emotion_label', order=EMOTION_NAMES, ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Emotion Distribution')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=metadata, x='emotion_label', hue='gender', order=EMOTION_NAMES, ax=axes[1], palette='Set1')
axes[1].set_title('Emotion Distribution by Gender')
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Gender')

plt.tight_layout()
plt.show()

In [ ]:
sample_video = metadata.iloc[0]['filepath']
cap = cv2.VideoCapture(sample_video)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

indices = np.linspace(0, total_frames - 1, 8, dtype=int)
frames = []
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

fig, axes = plt.subplots(1, len(frames), figsize=(20, 3))
sample_meta = metadata.iloc[0]
fig.suptitle(f'Sample: {sample_meta["emotion_label"]} (Actor {sample_meta["actor"]}, {sample_meta["gender"]})', fontsize=14)
for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.imshow(frame)
    ax.set_title(f'Frame {indices[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
stats = []
for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc='Scanning videos'):
    cap = cv2.VideoCapture(row['filepath'])
    stats.append({
        'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    })
    cap.release()

stats_df = pd.DataFrame(stats)
stats_df['duration'] = stats_df['frames'] / stats_df['fps']
print('Video Statistics:')
print(stats_df[['frames', 'fps', 'duration', 'width', 'height']].describe().round(2))

## 4. Face Detection & Preprocessing

For each video we:
1. Sample 15 uniformly-spaced frames
2. Detect and crop faces using MTCNN (with margin for context)
3. If a frame has no detection, fall back to the nearest detected bounding box
4. Save the result as a `.npy` array of shape `(15, 224, 224, 3)`

In [ ]:
mtcnn = MTCNN(
    image_size=CONFIG['face_size'],
    margin=40,
    min_face_size=60,
    thresholds=[0.6, 0.7, 0.7],
    post_process=False,
    device=device,
)

In [ ]:
def sample_frames(video_path, num_frames):
    """Sample num_frames evenly spaced frames from a video as PIL Images."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    
    indices = np.linspace(0, total - 1, min(num_frames, total), dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
    cap.release()
    
    # Pad with last frame if needed
    while len(frames) < num_frames and len(frames) > 0:
        frames.append(frames[-1])
    
    return frames[:num_frames]


def extract_faces(frames, detector, face_size=224):
    """Detect and crop faces from PIL frames with temporal fallback."""
    faces = [None] * len(frames)
    boxes = [None] * len(frames)
    
    for i, frame in enumerate(frames):
        detected_boxes, probs = detector.detect(frame)
        if detected_boxes is not None and len(detected_boxes) > 0:
            boxes[i] = detected_boxes[0]  # Take highest-confidence face
    
    detected_indices = [i for i, b in enumerate(boxes) if b is not None]
    if not detected_indices:
        return None
    
    # Fill missing detections with nearest detected box
    for i in range(len(frames)):
        if boxes[i] is None:
            nearest = min(detected_indices, key=lambda j: abs(j - i))
            boxes[i] = boxes[nearest]
    
    for i, frame in enumerate(frames):
        img = np.array(frame)
        x1, y1, x2, y2 = [int(c) for c in boxes[i]]
        # Clamp to image bounds
        h, w = img.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            # Fallback: use center crop
            cx, cy = w // 2, h // 2
            half = min(w, h) // 3
            x1, y1 = cx - half, cy - half
            x2, y2 = cx + half, cy + half
        crop = img[y1:y2, x1:x2]
        face = cv2.resize(crop, (face_size, face_size))
        faces[i] = face
    
    return np.stack(faces)  # (num_frames, face_size, face_size, 3)


def preprocess_video(video_path, detector, num_frames=15, face_size=224):
    """Full pipeline: sample frames -> detect faces -> return array."""
    frames = sample_frames(video_path, num_frames)
    if not frames:
        return None
    return extract_faces(frames, detector, face_size)

In [ ]:
failures = []

for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc='Preprocessing videos'):
    video_name = Path(row['filepath']).stem
    npy_path = PROCESSED_DIR / f'{video_name}.npy'
    
    if npy_path.exists():
        continue
    
    faces = preprocess_video(
        row['filepath'], mtcnn,
        num_frames=CONFIG['num_frames'],
        face_size=CONFIG['face_size'],
    )
    
    if faces is not None:
        np.save(npy_path, faces.astype(np.uint8))
    else:
        failures.append(row['filepath'])

print(f'\nProcessed: {len(metadata) - len(failures)} / {len(metadata)}')
print(f'Failures: {len(failures)}')
if failures:
    print('Failed files:', failures[:5])

In [ ]:
fig, axes = plt.subplots(3, CONFIG['num_frames'], figsize=(20, 5))

sample_emotions = [3, 5, 8]  # happy, angry, surprised
for row_idx, emo in enumerate(sample_emotions):
    sample = metadata[metadata['emotion'] == emo].iloc[0]
    npy_path = PROCESSED_DIR / f"{Path(sample['filepath']).stem}.npy"
    if not npy_path.exists():
        continue
    faces = np.load(npy_path)
    for col_idx in range(CONFIG['num_frames']):
        axes[row_idx, col_idx].imshow(faces[col_idx])
        axes[row_idx, col_idx].axis('off')
    axes[row_idx, 0].set_ylabel(EMOTION_MAP[emo], fontsize=12, rotation=0, labelpad=60)

plt.suptitle('Extracted Face Sequences', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Dataset & DataLoader

**Actor-disjoint split** prevents the model from learning actor identity instead of emotion:
- **Train**: Actors 1–18 (75%)
- **Validation**: Actors 19–22 (~17%)
- **Test**: Actors 23–24 (~8%)

In [ ]:
TRAIN_ACTORS = list(range(1, 19))
VAL_ACTORS = list(range(19, 23))
TEST_ACTORS = list(range(23, 25))

metadata['npy_path'] = metadata['filepath'].apply(
    lambda f: str(PROCESSED_DIR / f"{Path(f).stem}.npy")
)
metadata_valid = metadata[metadata['npy_path'].apply(os.path.exists)].reset_index(drop=True)

print(f'Total metadata rows: {len(metadata)}, with existing .npy files: {len(metadata_valid)}')
if len(metadata_valid) == 0:
    raise RuntimeError(
        f"No .npy files found. Check that preprocessing (face detection) ran successfully "
        f"and saved files to {PROCESSED_DIR}"
    )

train_df = metadata_valid[metadata_valid['actor'].isin(TRAIN_ACTORS)].reset_index(drop=True)
val_df = metadata_valid[metadata_valid['actor'].isin(VAL_ACTORS)].reset_index(drop=True)
test_df = metadata_valid[metadata_valid['actor'].isin(TEST_ACTORS)].reset_index(drop=True)

print(f'Train: {len(train_df)} samples ({len(TRAIN_ACTORS)} actors)')
print(f'Val:   {len(val_df)} samples ({len(VAL_ACTORS)} actors)')
print(f'Test:  {len(test_df)} samples ({len(TEST_ACTORS)} actors)')

assert len(val_df) > 0, "Validation set is empty! .npy files missing for actors 19-22. Re-run preprocessing."
assert len(test_df) > 0, "Test set is empty! .npy files missing for actors 23-24. Re-run preprocessing."

print(f'\nTrain emotion distribution:')
print(train_df['emotion_label'].value_counts().sort_index())

In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class RAVDESSDataset(Dataset):
    def __init__(self, df, transform=None, num_frames=15):
        self.samples = list(zip(df['npy_path'].values, df['label'].values))
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        npy_path, label = self.samples[idx]
        faces = np.load(npy_path)  # (num_frames, 224, 224, 3) uint8

        frames = []
        for i in range(self.num_frames):
            frame = faces[i]  # (224, 224, 3)
            if self.transform:
                frame = self.transform(frame)
            else:
                frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
            frames.append(frame)

        frames = torch.stack(frames)  # (num_frames, 3, 224, 224)
        label = torch.tensor(label, dtype=torch.long)
        return frames, label

In [ ]:
train_dataset = RAVDESSDataset(train_df, train_transform, CONFIG['num_frames'])
val_dataset = RAVDESSDataset(val_df, val_transform, CONFIG['num_frames'])
test_dataset = RAVDESSDataset(test_df, val_transform, CONFIG['num_frames'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

batch_frames, batch_labels = next(iter(train_loader))
print(f'Batch frames shape: {batch_frames.shape}')   # (B, 15, 3, 224, 224)
print(f'Batch labels shape: {batch_labels.shape}')    # (B,)
print(f'Label values: {batch_labels.tolist()}')

## 6. Model Architecture

```
Input: (B, T=15, 3, 224, 224)
         │
         ▼
  ResNet-18 (per-frame, layers 1-2 frozen, layers 3-4 trainable) ──► (B, T, 512)
         │
         ▼
  Bidirectional LSTM (2 layers, hidden=256) ──► (B, T, 512)
         │
         ▼
  Temporal Attention ──► (B, 512)  [weighted sum across timesteps]
         │
         ▼
  FC(512 → 8) ──► logits
```

**Key changes from baseline:**
- **Unfrozen layer3 & layer4**: Allows the CNN to adapt its spatial features for facial expression recognition rather than relying solely on generic ImageNet features.
- **Bidirectional 2-layer LSTM**: Captures temporal dynamics in both forward and backward directions, enabling the model to use future context when interpreting each frame.
- **Temporal Attention**: Instead of only using the last hidden state, the model learns to attend to the most emotionally salient frames in the sequence.
- **Discriminative learning rates**: Lower LR (1e-5) for pretrained CNN layers, higher LR (1e-4) for the LSTM and classification head.

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """ResNet-18 backbone, layers 1-2 frozen, layers 3-4 trainable."""

    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC, output (B, 512, 1, 1)

        # Freeze early layers: conv1, bn1, relu, maxpool, layer1, layer2 (indices 0-5)
        # Keep layer3 (index 6) and layer4 (index 7) trainable
        for i, child in enumerate(self.features.children()):
            if i <= 5:
                for param in child.parameters():
                    param.requires_grad = False

    def forward(self, x):
        x = self.features(x)       # (B, 512, 1, 1)
        x = x.view(x.size(0), -1)  # (B, 512)
        return x


class TemporalAttention(nn.Module):
    """Attention mechanism over temporal dimension."""

    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, lstm_out):
        # lstm_out: (B, T, hidden_size)
        scores = self.attention(lstm_out)        # (B, T, 1)
        weights = F.softmax(scores, dim=1)       # (B, T, 1)
        context = (lstm_out * weights).sum(dim=1) # (B, hidden_size)
        return context


class EmotionRecognitionModel(nn.Module):
    """CNN + Bidirectional LSTM + Temporal Attention for video emotion recognition."""

    def __init__(self, num_classes=8, lstm_hidden=256, lstm_layers=2, dropout=0.5):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        self.feature_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )
        self.attention = TemporalAttention(lstm_hidden * 2)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        # Process all frames through CNN at once
        x = x.view(B * T, C, H, W)           # (B*T, 3, 224, 224)
        features = self.cnn(x)                 # (B*T, 512)
        features = features.view(B, T, -1)     # (B, T, 512)
        features = self.feature_dropout(features)
        lstm_out, _ = self.lstm(features)       # (B, T, 512) bidirectional
        context = self.attention(lstm_out)      # (B, 512)
        out = self.dropout(context)
        out = self.fc(out)                      # (B, 8)
        return out

In [ ]:
model = EmotionRecognitionModel(
    num_classes=CONFIG['num_classes'],
    lstm_hidden=CONFIG['lstm_hidden'],
    lstm_layers=CONFIG['lstm_layers'],
    dropout=CONFIG['dropout'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters:    {total_params - trainable_params:,}')

In [ ]:
train_labels = train_df['label'].values
class_weights = compute_class_weight('balanced', classes=np.arange(CONFIG['num_classes']), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:', {EMOTION_NAMES[i]: f'{w:.2f}' for i, w in enumerate(class_weights.cpu().numpy())})

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Discriminative learning rates: lower for pretrained CNN, higher for LSTM/attention/FC
cnn_params = [p for p in model.cnn.parameters() if p.requires_grad]
other_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('cnn')]

optimizer = torch.optim.AdamW([
    {'params': cnn_params, 'lr': CONFIG['cnn_learning_rate']},
    {'params': other_params, 'lr': CONFIG['learning_rate']},
], weight_decay=CONFIG['weight_decay'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

print(f'CNN params (lr={CONFIG["cnn_learning_rate"]}): {sum(p.numel() for p in cnn_params):,}')
print(f'Other params (lr={CONFIG["learning_rate"]}): {sum(p.numel() for p in other_params):,}')

## 7. Training

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=1.0):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for frames, labels in tqdm(loader, desc='Train', leave=False):
        frames, labels = frames.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        running_loss += loss.item() * frames.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    if total == 0:
        return 0.0, 0.0
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for frames, labels in tqdm(loader, desc='Eval', leave=False):
        frames, labels = frames.to(device), labels.to(device)

        outputs = model(frames)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * frames.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    if total == 0:
        raise ValueError(
            "Validation loader is empty (0 samples). "
            "Check that .npy files exist for validation actors (19-22). "
            "Re-run the preprocessing cell (face detection) and verify the files were saved."
        )
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
best_epoch = 0

for epoch in range(1, CONFIG['num_epochs'] + 1):
    print(f'\nEpoch {epoch}/{CONFIG["num_epochs"]}')
    print('-' * 40)

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, CONFIG['grad_clip']
    )
    val_loss, val_acc, val_preds, val_labels = evaluate(
        model, val_loader, criterion, device
    )

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_acc)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_model.pth')
        print(f'  >> New best model saved (val_acc={val_acc:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['early_stop_patience']:
            print(f'  >> Early stopping at epoch {epoch}')
            break

    torch.cuda.empty_cache()

print(f'\nBest validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train')
ax1.plot(epochs_range, history['val_loss'], label='Validation')
ax1.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], label='Train')
ax2.plot(epochs_range, history['val_acc'], label='Validation')
ax2.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluation

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_DIR / 'best_model.pth', map_location=device, weights_only=True))
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss:     {test_loss:.4f}')
print(f'\nTest F1 (macro):    {f1_score(test_labels, test_preds, average="macro"):.4f}')
print(f'Test F1 (weighted): {f1_score(test_labels, test_preds, average="weighted"):.4f}')

In [ ]:
print('Classification Report:')
print('=' * 60)
print(classification_report(test_labels, test_preds, target_names=EMOTION_NAMES, digits=3))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_NAMES,
            yticklabels=EMOTION_NAMES, ax=ax1)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_title('Confusion Matrix (Counts)')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=EMOTION_NAMES,
            yticklabels=EMOTION_NAMES, ax=ax2)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(EMOTION_NAMES, per_class_acc, color=plt.cm.Set2(np.arange(8)))
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Test Accuracy')
ax.set_ylim(0, 1.0)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{acc:.2f}', ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
test_meta = test_df.copy()
test_meta['predicted'] = test_preds
test_meta['correct'] = (test_meta['label'] == test_meta['predicted']).astype(int)

gender_acc = test_meta.groupby('gender')['correct'].mean()
print('Accuracy by Gender:')
for gender, acc in gender_acc.items():
    print(f'  {gender}: {acc:.4f}')

gender_emo_acc = test_meta.groupby(['gender', 'emotion_label'])['correct'].mean().unstack(fill_value=0)
gender_emo_acc.plot(kind='bar', figsize=(12, 5), rot=0)
plt.title('Accuracy by Gender and Emotion')
plt.ylabel('Accuracy')
plt.legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Qualitative Analysis

In [ ]:
correct_mask = test_preds == test_labels
incorrect_mask = ~correct_mask


def show_predictions(indices, title):
    n = min(4, len(indices))
    if n == 0:
        print(f'No samples for: {title}')
        return
    fig, axes = plt.subplots(n, CONFIG['num_frames'], figsize=(20, 3 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(title, fontsize=14)

    for row_idx, sample_idx in enumerate(indices[:n]):
        npy_path = test_df.iloc[sample_idx]['npy_path']
        faces = np.load(npy_path)
        true_label = EMOTION_NAMES[test_labels[sample_idx]]
        pred_label = EMOTION_NAMES[test_preds[sample_idx]]

        for col_idx in range(CONFIG['num_frames']):
            axes[row_idx, col_idx].imshow(faces[col_idx])
            axes[row_idx, col_idx].axis('off')
        axes[row_idx, 0].set_ylabel(
            f'True: {true_label}\nPred: {pred_label}',
            fontsize=10, rotation=0, labelpad=80
        )

    plt.tight_layout()
    plt.show()


correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(incorrect_mask)[0]

show_predictions(correct_indices, 'Correctly Classified Samples')
show_predictions(incorrect_indices, 'Misclassified Samples')

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    """Extract LSTM last hidden states for all samples."""
    model.eval()
    embeddings = []
    labels = []

    for frames, batch_labels in tqdm(loader, desc='Extracting embeddings'):
        frames = frames.to(device)
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)
        features = model.cnn(x)
        features = features.view(B, T, -1)
        lstm_out, _ = model.lstm(features)
        last_hidden = lstm_out[:, -1, :]  # (B, 256)

        embeddings.append(last_hidden.cpu().numpy())
        labels.append(batch_labels.numpy())

    return np.concatenate(embeddings), np.concatenate(labels)


test_embeddings, test_embed_labels = extract_embeddings(model, test_loader, device)

tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=min(30, len(test_embeddings) - 1))
embeddings_2d = tsne.fit_transform(test_embeddings)

plt.figure(figsize=(10, 8))
for i, emotion in enumerate(EMOTION_NAMES):
    mask = test_embed_labels == i
    if mask.any():
        plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], label=emotion, alpha=0.7, s=50)
plt.legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title('t-SNE of LSTM Embeddings (Test Set)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## 10. Discussion & Conclusion

### Results Summary

The CNN+LSTM model leverages both spatial features (facial structure captured by ResNet-18) and temporal dynamics (expression evolution captured by LSTM) for video-based emotion recognition on RAVDESS.

**Key observations:**
- The actor-disjoint split provides a realistic generalization estimate — the model must recognize emotions from actors never seen during training
- Certain emotion pairs (e.g., calm vs. neutral, fear vs. surprise) are commonly confused, consistent with their visual similarity
- The temporal component (LSTM) helps distinguish emotions that share similar peak expressions but differ in onset/offset dynamics

### Comparison to Prior Work

State-of-the-art on RAVDESS video-only emotion recognition typically achieves 60–75% accuracy depending on methodology and evaluation protocol. Our CNN+LSTM baseline provides a strong foundation.

### Limitations
- **Small test set**: Only 2 actors — results may have high variance
- **Actor pool**: 24 actors from one cultural background limits diversity
- **Acted emotions**: May not generalize well to spontaneous/naturalistic expressions
- **Single modality**: Uses only visual information; audio contains complementary cues

### Potential Extensions
- **Attention mechanisms**: Temporal attention over LSTM outputs (instead of just using the last hidden state) to weight important frames
- **Multimodal fusion**: Combine video features with audio features (MFCCs, spectrograms) for richer representation
- **Larger backbones**: EfficientNet-B4, ResNet-50, or Vision Transformers for stronger spatial features
- **Transformer temporal modeling**: Replace LSTM with a Transformer encoder for better long-range dependencies
- **Cross-dataset evaluation**: Test on FER2013, AffectNet, or CK+ to assess generalization
- **Real-time inference**: Optimize the pipeline for webcam-based real-time emotion recognition
- **Emotion intensity estimation**: Regress continuous intensity values instead of discrete classification

### Applications
- Human-computer interaction and adaptive interfaces
- Mental health monitoring and therapy support
- Education technology (engagement detection)
- Entertainment and gaming

### References

1. Livingstone, S. R., & Russo, F. A. (2018). The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS). *PLoS ONE*, 13(5), e0196391.
2. He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*.
3. Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.
4. Zhang, K., Zhang, Z., Li, Z., & Qiao, Y. (2016). Joint face detection and alignment using multitask cascaded convolutional networks. *IEEE Signal Processing Letters*, 23(10), 1499–1503.